# إعادة التحليل الكامل على البيانات النظيفة (محدَّثة حتى الآن)

هذا النوتبوك يعيد كل التحليلات الأصلية (VADER، LDA، التحليل الزمني) على البيانات الصحيحة بعد إصلاح مشكلة التكرار، ويضيف تحليل انتشار المواضيع عبر الزمن (كان الهدف الأصلي قبل اكتشاف المشكلة).

**ارفعي ملف `reddit_chatgpt_posts_clean.csv` إلى بيئة Colab أولًا** (أيقونة المجلد 📁 يسار الشاشة → Upload → اختاري الملف)، ثم شغّلي الخلايا بالترتيب.


## 1. التثبيت والتحميل

In [ ]:
!pip install vaderSentiment gensim -q

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')


In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("reddit_chatgpt_posts_clean.csv", low_memory=False)
print(f"إجمالي الصفوف: {len(df):,}")
print(f"معرّفات فريدة: {df['id'].nunique():,}")
assert df['id'].nunique() == len(df), "⚠️ ما زال هناك تكرار! توقفي وتحققي."
print("✅ لا يوجد تكرار — البيانات نظيفة")

df['date'] = pd.to_datetime(df['created_utc'], errors='coerce')
print(f"النطاق الزمني: {df['date'].min()} إلى {df['date'].max()}")


## 2. المعالجة الأولية للنص (مطابقة لمنهجية الورقة الأصلية، القسم 3.2)

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# 1. دمج العنوان والنص (يحافظ على السياق الكامل)
df['merged_text'] = df['title'].fillna('') + " " + df['text'].fillna('')

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return ' '.join(tokens)

print(f"جاري تنظيف النصوص ({len(df):,} منشورًا، قد يستغرق بضع دقائق)...")
df['cleaned_text'] = df['merged_text'].apply(preprocess_text)
print("✅ اكتمل التنظيف")
df[['id','merged_text','cleaned_text']].head(3)


## 3. تحليل المشاعر (VADER)

**تنبيه منهجي مهم**: النسخة الأصلية طبّقت VADER على `cleaned_text` (بعد إزالة كلمات التوقف). لكن إزالة كلمات مثل "not" أو "no" أو "never" (تُصنَّف كـ stopwords) يمكن أن تُحرّف قراءة السياق العاطفي — مثال: "not good" (سلبية) تصبح "good" (إيجابية) بعد إزالة "not". VADER مصمم أصلًا للعمل على نص طبيعي غير معالَج (يستفيد من علامات الترقيم، الأحرف الكبيرة، والنفي).

لذلك نُشغّل VADER هنا على **النص الخام (`merged_text`) كطريقة أساسية أدق منهجيًا**، وأيضًا على `cleaned_text` كنسخة للمقارنة مع الأرقام المنشورة سابقًا في الورقة.


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    return analyzer.polarity_scores(str(text))['compound']

def categorize(score):
    if score >= 0.05: return 'positive'
    elif score <= -0.05: return 'negative'
    else: return 'neutral'

print("جاري تحليل المشاعر على النص الخام (الطريقة الأساسية الموصى بها)...")
df['sentiment_score_raw'] = df['merged_text'].apply(get_sentiment)
df['sentiment_raw'] = df['sentiment_score_raw'].apply(categorize)

print("جاري تحليل المشاعر على النص المُنظَّف (للمقارنة مع الورقة الأصلية)...")
df['sentiment_score_cleaned'] = df['cleaned_text'].apply(get_sentiment)
df['sentiment_cleaned'] = df['sentiment_score_cleaned'].apply(categorize)

df.to_csv("sentiment_analysis_results_clean.csv", index=False)
print("\n✅ تم الحفظ في sentiment_analysis_results_clean.csv")


In [ ]:
print("="*60)
print("توزيع المشاعر — النص الخام (الطريقة الموصى بها):")
print(df['sentiment_raw'].value_counts())
print((df['sentiment_raw'].value_counts(normalize=True)*100).round(1))

print("\n" + "="*60)
print("توزيع المشاعر — النص المُنظَّف (للمقارنة مع أرقام الورقة القديمة):")
print(df['sentiment_cleaned'].value_counts())
print((df['sentiment_cleaned'].value_counts(normalize=True)*100).round(1))


## 4. التحليل الزمني للمشاعر (شهري)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df['month_year'] = df['date'].dt.to_period('M')
monthly_counts = df.groupby(['month_year', 'sentiment_raw']).size().unstack(fill_value=0)
monthly_pct = monthly_counts.div(monthly_counts.sum(axis=1), axis=0) * 100
monthly_pct = monthly_pct.reset_index()
monthly_pct['month_year'] = monthly_pct['month_year'].astype(str)
monthly_pct = monthly_pct.sort_values('month_year')

# عدد المنشورات لكل شهر (لتقييم موثوقية كل نقطة)
monthly_n = df.groupby('month_year').size()
print("عدد المنشورات لكل شهر (تحققي أن الأشهر ذات العدد القليل جدًا تُفسَّر بحذر):")
print(monthly_n.to_string())

plt.figure(figsize=(14, 6))
sns.set_style("whitegrid")
colors = {'positive': '#2ecc71', 'neutral': '#95a5a6', 'negative': '#e74c3c'}
for s in ['positive', 'neutral', 'negative']:
    if s in monthly_pct.columns:
        plt.plot(monthly_pct['month_year'], monthly_pct[s], marker='o', linewidth=2, label=s.capitalize(), color=colors[s])
plt.xlabel('Month'); plt.ylabel('Percentage (%)')
plt.title('Temporal Evolution of Public Sentiment Toward ChatGPT on Reddit — CLEAN DATA (Nov 2022 – Mar 2025)')
plt.xticks(rotation=45, ha='right'); plt.legend(); plt.tight_layout()
plt.savefig('sentiment_evolution_clean.png', dpi=300, bbox_inches='tight')
plt.show()

monthly_pct.to_csv("monthly_sentiment_clean.csv", index=False)
print("\n✅ تم الحفظ في monthly_sentiment_clean.csv")


## 5. تنقيح العينة لنمذجة المواضيع (مطابق لمنهجية الورقة)

نفس معايير الترشيح الأصلية:
1. فلتر التفاعل: score ≥ 5
2. فلتر الطول: ≥ 50 كلمة بعد التنظيف
3. إزالة [deleted]/[removed]


In [ ]:
df['score'] = pd.to_numeric(df['score'], errors='coerce')
print(f"البداية: {len(df):,} منشورًا")

df_eng = df[df['score'] >= 5]
print(f"1️⃣ بعد فلتر التفاعل (score≥5): {len(df_eng):,} (أُزيل {len(df)-len(df_eng):,})")

df_len = df_eng.copy()
df_len['word_count'] = df_len['cleaned_text'].str.split().str.len()
df_len = df_len[df_len['word_count'] >= 50]
print(f"2️⃣ بعد فلتر الطول (≥50 كلمة): {len(df_len):,} (أُزيل {len(df_eng)-len(df_len):,})")

mask_deleted = df_len['merged_text'].astype(str).str.contains(r'\[deleted\]|\[removed\]', case=False, na=False)
df_refined = df_len[~mask_deleted].copy()
print(f"3️⃣ بعد إزالة [deleted]/[removed]: {len(df_refined):,} (أُزيل {len(df_len)-len(df_refined):,})")

print(f"\n✅ العينة المُنقّحة النهائية لنمذجة المواضيع: {len(df_refined):,} منشورًا")
df_refined.to_csv("refined_corpus_clean.csv", index=False)


## 6. إزالة الكلمات عالية التكرار غير الدلالية + اكتشاف الثنائيات (bigrams)

In [ ]:
domain_specific_noise = [
    "chatgpt", "ai", "gpt", "llm", "model", "use", "like", "im", "ive",
    "response", "prompt", "data", "language", "tool", "openai", "google",
    "human", "user", "question", "paper", "llama", "openais", "deepseek",
    "perplexity", "chrome", "app", "apps", "code",
]
generic_terms = [
    "new", "using", "know", "think", "help", "really", "make", "want",
    "say", "used", "need", "time", "work", "also", "even", "still",
    "many", "much", "well", "way", "thing", "something", "anything",
]
combined_noise = set(stop_words).union(domain_specific_noise).union(generic_terms)

def remove_noise(text):
    tokens = str(text).split()
    return [t for t in tokens if t not in combined_noise and len(t) > 2]

texts = df_refined['cleaned_text'].apply(remove_noise).tolist()
texts = [t for t in texts if len(t) > 0]
print(f"عدد المستندات الصالحة لنمذجة المواضيع: {len(texts):,}")

from gensim.models import Phrases
from gensim.models.phrases import Phraser
bigram_model = Phrases(texts, min_count=5, threshold=10)
bigram_phraser = Phraser(bigram_model)
texts = [bigram_phraser[t] for t in texts]
print("✅ تم اكتشاف الثنائيات (مثل prompt_engineering، data_privacy)")


## 7. بناء القاموس وتدريب نماذج LDA بأعداد مواضيع مختلفة (5، 10، 15) مع مقارنة التماسك

In [ ]:
from gensim import corpora, models
from gensim.models import CoherenceModel

dictionary = corpora.Dictionary(texts)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(t) for t in texts]

topic_counts = [5, 10, 15]
comparison_results = []
lda_models = {}

for k in topic_counts:
    print(f"📊 تدريب LDA بـ {k} مواضيع...")
    lda_model = models.LdaModel(
        corpus=corpus, id2word=dictionary, num_topics=k,
        passes=20, random_state=42, per_word_topics=True,
    )
    coherence_model = CoherenceModel(model=lda_model, texts=texts, dictionary=dictionary, coherence='c_v')
    coherence_score = coherence_model.get_coherence()
    lda_models[k] = lda_model
    comparison_results.append({"num_topics": k, "coherence_score": round(coherence_score, 4)})
    print(f"   Coherence (Cv) = {coherence_score:.4f}")

results_df = pd.DataFrame(comparison_results)
print("\n📈 جدول المقارنة (لتحديث Table 4 في الورقة):")
print(results_df.to_string(index=False))
results_df.to_csv("lda_coherence_comparison_clean.csv", index=False)


## 8. اختيار أفضل عدد مواضيع

راجعي جدول التماسك أعلاه ثم عدّلي `BEST_K` في الخلية التالية للعدد الذي تختارينه (الورقة الأصلية اختارت 10 بناءً على التوازن بين التماسك وقابلية التفسير البشري — بما أن حجم العينة تغيّر، قد يكون عدد مختلف أنسب الآن، راجعي الكلمات المفتاحية العليا لكل عدد قبل القرار).


In [ ]:
BEST_K = 10  # ✏️ عدّلي هذا الرقم بعد مراجعة جدول التماسك أعلاه

best_model = lda_models[BEST_K]
print(f"المواضيع الـ {BEST_K} المختارة، أهم 15 كلمة لكل موضوع:\n")
for i in range(BEST_K):
    words = [w for w, p in best_model.show_topic(i, topn=15)]
    print(f"Topic #{i}: {', '.join(words)}")


## 9. الموضوع المهيمن لكل مستند وتوزيعه

In [ ]:
doc_topics = [best_model.get_document_topics(bow) for bow in corpus]
dominant_topics = [max(dt, key=lambda x: x[1])[0] if dt else -1 for dt in doc_topics]

df_topics = df_refined.iloc[:len(dominant_topics)].copy()
df_topics['dominant_topic'] = dominant_topics

print("توزيع المواضيع عبر المستندات:")
topic_dist = df_topics['dominant_topic'].value_counts().sort_index()
print(topic_dist)
print((topic_dist / len(df_topics) * 100).round(2))

plt.figure(figsize=(10,5))
topic_dist.plot(kind='bar', color='#4C72B0', edgecolor='black')
plt.xlabel('Topic Number'); plt.ylabel('Number of Documents')
plt.title('Distribution of Dominant Topics Across Documents — CLEAN DATA')
plt.tight_layout()
plt.savefig('topic_distribution_clean.png', dpi=300, bbox_inches='tight')
plt.show()

df_topics.to_csv("documents_with_topics_clean.csv", index=False)
print("\n✅ تم الحفظ في documents_with_topics_clean.csv")


## 10. جديد: انتشار المواضيع عبر الزمن (Topic-over-time)

هذا هو التحليل الذي طلبتِه أصلًا في بداية عملية المراجعة — يُظهر كيف تغيّر الاهتمام بكل موضوع شهرًا بشهر، ويحوّل ادّعاء الورقة عن "topic prevalence over time" من ادّعاء غير مُثبَت إلى نتيجة فعلية.


In [ ]:
df_topics['month_year'] = pd.to_datetime(df_topics['created_utc'], errors='coerce').dt.to_period('M').astype(str)

topic_month = df_topics.groupby(['month_year', 'dominant_topic']).size().unstack(fill_value=0)
topic_month_pct = topic_month.div(topic_month.sum(axis=1), axis=0) * 100
topic_month_pct = topic_month_pct.sort_index()

plt.figure(figsize=(16, 8))
sns.heatmap(topic_month_pct.T, cmap='YlOrRd', annot=False, cbar_kws={'label': '% of monthly posts'})
plt.xlabel('Month'); plt.ylabel('Topic Number')
plt.title('Topic Prevalence Over Time — CLEAN DATA (Nov 2022 – Mar 2025)')
plt.tight_layout()
plt.savefig('topic_over_time_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

topic_month_pct.to_csv("topic_prevalence_over_time.csv")
print("✅ تم الحفظ في topic_prevalence_over_time.csv")
print("\nمعاينة الجدول:")
print(topic_month_pct.round(1))


## 11. تنزيل كل الملفات الناتجة

In [ ]:
from google.colab import files
import os

output_files = [
    "sentiment_analysis_results_clean.csv",
    "monthly_sentiment_clean.csv",
    "refined_corpus_clean.csv",
    "lda_coherence_comparison_clean.csv",
    "documents_with_topics_clean.csv",
    "topic_prevalence_over_time.csv",
    "sentiment_evolution_clean.png",
    "topic_distribution_clean.png",
    "topic_over_time_heatmap.png",
]
for f in output_files:
    if os.path.exists(f):
        files.download(f)
    else:
        print(f"⚠️ لم يُنشأ بعد: {f}")
